# **수상작 리뷰 — 와인 품질 분류**

- 주제: 화학 성분 11가지 + 와인 종류로 품질 등급 예측
  - 품질 등급(3~9등급, 7단계)
- 평가 지표: 정확도(Accuracy)
- 최종 점수: 리더보드 0.713 - 랜덤 포레스트 + ExtraTrees + 스태킹 앙상블 + TabCL 결합


# 1. 데이터
- train, test : 와인 성분 및 품질 데이터
  - 학습 데이터 : 5497행 x 14열
  - 테스트 데이터 : 1000행 x 13열(quality 제외)
  - 모든 변수에 대해 결측치 없음
- sample_submission : 제출 파일 예시
- 데이터 특징 - EDA
  - 5~7등급이 전체의 93.3% / 정확도 : 0.4395 (기준)
  - 학습/테스트 분포가 거의 같음 (평균 차이 3% 이내, white 비율 0.757 vs 0.739) → 분포 보정 불필요
  - 학습 데이터의 15.6%(858행)가 중복, 테스트 1000행 중 303행이 학습 데이터와 성분이 완전히 같음
  - 성분이 같은데 품질이 다른 경우는 0건

# 2. 코드 흐름

- 1. 데이터 이해
  - 데이터 구조·결측치 확인, 품질 등급 분포 확인 (5~7등급 편중)
  - 학습/테스트 분포 비교 -> 차이 거의 없음
  - 변수별 왜도·극단값 확인: chlorides(왜도 5.19) 등 오른쪽으로 치우친 변수가 많고, 극단값 대부분이 레드 와인
- 2. 변수 분석 & 전처리 결정
  - 상관관계: alcohol(0.44), density(-0.30)가 품질과 가장 관련 큼
    - 상관 강한 5개만 쓰면 정확도 0.668 → 0.642로 하락 -> 전체 변수 유지
  - 레드/화이트 분리 모델(0.6767) vs 통합 모델(0.6680) -> 차이가 표준편차 범위 내 -> is_white 변수 추가 후 통합 모델
  - 중복 제거 시 정확도 0.668 -> 0.577로 급락 + 테스트의 30%가 학습과 동일 -> 중복 유지
  - 클래스 불균형 처리(balanced, oversampling, SMOTE) -> 개선 미미 -> 처리X
  - 파생 변수 bound_so2(결합 이산화황 = 총-독립) 추가 -> 세 모델 모두 소폭 상승 -> 추가
- 3. 검증 설계와 모델
  - 일반 5-Fold CV는 같은 와인이 학습,검증 폴드에 동시에 들어가서 점수가 약 10%p 부풀려짐
  - 성분이 같은 행을 한 그룹으로 묶는 그룹 교차 검증으로 모델 비교
  - 로지스틱 회귀(0.54) vs 트리 모델(0.66~0.68) -> 비선형 관계 -> 트리 기반 모델 선택

- 4. 앙상블과 최종 모델
  - RandomForest + ExtraTrees + 스태킹(RF·ET·CatBoost → 로지스틱 회귀) 확률 평균 -> 리더보드: 0.709
  - 이후 트리 수, 시드, 비중을 바꾸거나 3단계 스태킹을 해도 0.709를 넘지 못함
  - 원인 분석: 트리 모델끼리 같은 행에서 함께 틀림 -> 원리가 다른 TabICL 결합
  - 최종: 0.709 확률 + 0.5xTabICL 확률 -> 리더보드: 0.713

# 3. 주요 코드 분석

1) 중복 데이터 확인

In [ ]:
key_cols = features + ['type']   # 11개 성분 + 와인 종류를 '같은 와인' 판단 기준으로 사용

# 학습 데이터 안에서 성분이 완전히 같은 행의 개수
duplicate_count = train.duplicated(subset=key_cols).sum()

# 테스트 데이터 중 학습 데이터와 성분이 똑같은 행의 개수 (inner merge)
overlap_count = test[key_cols].merge(train[key_cols].drop_duplicates(), how='inner').shape[0]

# 성분은 같은데 품질 등급이 2개 이상인 그룹의 개수
conflict_count = (train.groupby(key_cols)['quality'].nunique() > 1).sum()

print(f'학습 데이터 중복 행: {duplicate_count}개 ({duplicate_count / len(train):.1%})')
print(f'테스트 중 학습 데이터와 성분이 같은 행: {overlap_count}개')
print(f'성분은 같고 품질이 다른 경우: {conflict_count}건')

# 출력
# 학습 데이터 중복 행: 858개 (15.6%)
# 테스트 중 학습 데이터와 성분이 같은 행: 303개
# 성분은 같고 품질이 다른 경우: 0건

- 학습 데이터 중복 행: 858개 (15.6%)
- 테스트 중 학습 데이터와 성분이 같은 행: 303개
- 성분은 같고 품질이 다른 경우: 0건

--> 중복행 유지

2. 스태킹 앙상블

In [ ]:
# 스태킹 안쪽에서도 그룹 3-Fold 사용 → 같은 와인이 OOF 예측에 새어 들어가는 데이터 누수 방지
inner_cv = StratifiedGroupKFold(3, shuffle=True, random_state=42)

# 1단계: RF, ET, CatBoost 각각의 OOF(Out-Of-Fold) 클래스별 확률 생성
stack_oof = [cross_val_predict(clone(m), X_common, y, cv=inner_cv,
                               groups=groups, method='predict_proba')
             for m in [rf, et, cat]]

# 2단계: 세 모델의 확률을 옆으로 붙여 메타 모델(로지스틱 회귀) 학습
meta = make_pipeline(StandardScaler(), LogisticRegression(C=0.3))   # C=0.3: 규제를 강하게 줘서 과적합 방지
meta.fit(np.hstack(stack_oof), y)

# 최종: RF, ET, 스태킹의 확률을 1:1:1로 평균
proba_709 = (rf_proba + et_proba + stack_proba) / 3

- 그룹 3-Fold를 쓰는 이유 : 데이터 누수 문제의 발생
- 기준 모델로 정한 이유 : 그룹 교차검증 상위권 + 매번 같은 결과로 재현

3. 최종 모델 - TabCL 결합

In [ ]:
from tabicl import TabICLClassifier

# TabICL용 입력: 원본 성분 + is_white
X_raw = train[features].assign(is_white=(train['type'] == 'white').astype(int))

# 시드 3개로 예측한 확률을 평균 → 결과 안정화
tabicl_proba = sum(
    TabICLClassifier(n_estimators=8, random_state=seed, device='cpu')
        .fit(X_raw, y).predict_proba(X_raw_test)
    for seed in [42, 7, 2026]
) / 3

# 기존 0.709 확률에 TabICL 확률을 0.5 비중으로 더함
final_proba = proba_709 + 0.5 * tabicl_proba

# 확률이 가장 큰 클래스를 최종 예측 등급으로
pred = fitted[0].classes_[final_proba.argmax(axis=1)]

- TabICL 단독 그룹 교차검증: 0.584 (0.709 모델 0.582와 비슷)
- 0.709 모델과 예측이 다른 비율 : 10.5% (트리 모델은 6~7%)
  - 정확도는 비슷한데 틀리는 행이 다름
  - -> 서로의 오답 보완
- 리더보드 : 0.713

# 4. 새롭게 알게된 점

- 데이터 구조가 검증 방식을 결정한다
  - 중복 데이터가 많을 때 일반 KFold를 쓰면 점수가 10%p나 부풀려질 수 있다는 걸 알게 됐다
- 중복된다고 무조건 지우는 게 아니다
  - 보통 전처리에서 중복을 제거하지만, 테스트에도 같은 데이터가 있으면 유지하는 게 유리하다
  - 전처리 결정은 정답이 정해진 게 아니라 데이터 상황에 따라 검증해서 정해야 한다
- 효과가 없던 실험도 근거가 된다
  - 변수 제거, 레드/화이트 분리, SMOTE 같은 불균형 처리를 모두 실험해 보고, 효과가 없다는 결과를 근거로 "하지 않기"를 선택한 과정이 인상적이었다 - 특히 교차 검증 전에 SMOTE를 적용하면 데이터 누수가 생긴다는 점을 배웠다
- 앙상블은 '서로 다르게 틀리는' 모델을 섞어야 효과가 있다.
  - 트리 모델끼리는 같은 곳에서 틀려서 아무리 섞어도 점수가 오르지 않았고, 원리가 다른 TabICL을 넣었을 때 개선됐다
  - 모델 간 예측이 다른 비율을 확인하는 방법도 유용했다
- 데이터 자체의 한계
  - 성분이 거의 같아도 등급이 일치하는 비율은 절반 정도였다. 품질은 사람의 시음 평가라서 성분만으로는 예측에 한계가 있다